# 01 — Data Preprocessing

This notebook inspects raw datasets, runs each processor, and verifies the output schema before handing data to Part 2 (Embedding).

**Sections**
1. Configuration
2. Run processors
3. Schema verification
4. Class balance
5. Sample preview
6. Cross-dataset summary

In [ ]:
import sys
import os

# Make src importable from the notebooks/ folder
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from src.data_preprocessing import ToxigenProcessor, RussianProcessor

## 1. Configuration

Reads `configs/datasets.yaml`. Update paths there if your raw data lives elsewhere.

In [ ]:
CONFIG_PATH = "../configs/datasets.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

pre_cfg = cfg["preprocessing"]
OUTPUT_ROOT = pre_cfg["output_root"]
TEST_SIZE   = pre_cfg["test_size"]
RANDOM_STATE = pre_cfg["random_state"]

print("Output root :", OUTPUT_ROOT)
print("Test size   :", TEST_SIZE)
print("Random state:", RANDOM_STATE)

## 2. Run Processors

Each processor loads raw data, normalises it to `{text, group, label}` and saves `full.csv`, `train.csv`, `test.csv` under `outputs/1_preprocessed/{dataset}/`.

In [ ]:
# --- ToxiGen ---
tox_cfg = cfg["datasets"]["toxigen"]

toxigen_proc = ToxigenProcessor(
    data_dir=tox_cfg["data_dir"],
    groups=tox_cfg["groups"],
    output_root=OUTPUT_ROOT,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
tox_splits = toxigen_proc.process()

In [ ]:
# --- Russian ---
rus_cfg = cfg["datasets"]["russian"]

russian_proc = RussianProcessor(
    annotated_path=rus_cfg["annotated_path"],
    full_corpus_path=rus_cfg.get("full_corpus_path"),
    output_root=OUTPUT_ROOT,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
rus_splits = russian_proc.process()

## 3. Schema Verification

Confirm every output has exactly the three required columns with clean values.

In [ ]:
all_splits = {
    "toxigen": tox_splits,
    "russian": rus_splits,
}

for dataset, splits in all_splits.items():
    df = splits["full"]
    print(f"\n{'='*40}")
    print(f"Dataset : {dataset}")
    print(f"Columns : {df.columns.tolist()}")
    print(f"Shape   : {df.shape}")
    print(f"Dtypes  :\n{df.dtypes}")
    print(f"Nulls   : {df.isnull().sum().to_dict()}")
    print(f"Labels  : {sorted(df['label'].unique())}")
    print(f"Groups  : {sorted(df['group'].unique())}")

## 4. Class Balance

In [ ]:
fig, axes = plt.subplots(1, len(all_splits), figsize=(6 * len(all_splits), 4))
if len(all_splits) == 1:
    axes = [axes]

for ax, (dataset, splits) in zip(axes, all_splits.items()):
    counts = splits["full"]["label"].value_counts()
    ax.bar(counts.index, counts.values, color=["#e74c3c", "#2ecc71"])
    ax.set_title(f"{dataset} — label distribution")
    ax.set_ylabel("Count")
    for i, (lbl, cnt) in enumerate(counts.items()):
        ax.text(i, cnt + 5, str(cnt), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Per-group balance (ToxiGen)
df_tox = tox_splits["full"]
group_balance = (
    df_tox.groupby(["group", "label"])
    .size()
    .unstack(fill_value=0)
)
group_balance.plot(kind="bar", figsize=(14, 5), color=["#e74c3c", "#2ecc71"])
plt.title("ToxiGen — samples per group and label")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
group_balance

## 5. Sample Preview

In [ ]:
print("=== ToxiGen samples ===")
display(tox_splits["full"].sample(5, random_state=42)[["text", "group", "label"]])

print("\n=== Russian samples ===")
display(rus_splits["full"].sample(5, random_state=42)[["text", "group", "label"]])

## 6. Cross-Dataset Summary

In [ ]:
rows = []
for dataset, splits in all_splits.items():
    df = splits["full"]
    vc = df["label"].value_counts()
    rows.append({
        "dataset": dataset,
        "total": len(df),
        "train": len(splits["train"]),
        "test": len(splits["test"]),
        "hate": vc.get("hate", 0),
        "no_hate": vc.get("no_hate", 0),
        "hate_pct": round(vc.get("hate", 0) / len(df) * 100, 1),
        "n_groups": df["group"].nunique(),
    })

summary = pd.DataFrame(rows).set_index("dataset")
display(summary)